# 12. Context Enrichment - Window Around Chunk

Retrieve the best chunk, then add its neighbours so the LLM sees surrounding context.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

The best-matching chunk is often only a *piece* of the explanation; the sentence before or after holds the rest. **Window context enrichment** retrieves with small, precise chunks (good for matching) but sends the LLM the chunk plus its neighbours (good for understanding).

## How it works

1. Index small chunks with their sequence position
2. Find the top-k matching chunks
3. Expand each hit to indices `i-w ... i+w`
4. Merge overlapping windows, preserve document order
5. Send the expanded context to the LLM

```text
 chunks:  [c3] [c4] [c5*] [c6] [c7]      * = best match
 window=1  ------> send [c4][c5][c6] to the LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 12:23:51.896000 13956 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
def retrieve_with_window(query, k=2, window=1):
    hits = store.search(query, k)
    picked = set()
    for h in hits:
        picked.update(range(max(0, h["idx"] - window), min(len(chunks), h["idx"] + window + 1)))
    return [chunks[i] for i in sorted(picked)]   # keep original document order

query = "How does semantic chunking decide where to split?"
narrow = [h["text"] for h in store.search(query, 2)]
wide = retrieve_with_window(query, k=2, window=1)
print(f"{len(narrow)} chunks without window, {len(wide)} chunks with window")

2 chunks without window, 4 chunks with window


In [3]:
print("Without window:\n", answer(query, narrow), "\n")
print("With window:\n", answer(query, wide))

Without window:
 According to the context, semantic chunking places boundaries where the topic shifts, detected by comparing the embeddings of neighbouring sentences. 



With window:
 Semantic chunking decides where to split by comparing the embeddings of neighbouring sentences.


Because chunks overlap, adjacent chunks repeat text. For production, merge overlapping spans before prompting.

## Does it help retrieval or generation?
Windowing doesn't change *which chunk matches*, it changes *how much context the LLM gets*. Evidence that straddles a chunk boundary is the main beneficiary - check how often the evidence phrase appears in the windowed context but not in the single chunk.

In [4]:
import pandas as pd
rows = []
for q in load_eval_questions():
    single = [h["text"] for h in store.search(q["question"], 1)]
    windowed = retrieve_with_window(q["question"], k=1, window=1)
    rows.append({"question": q["question"][:50],
                 "in single chunk": any(q["evidence"].lower() in t.lower() for t in single),
                 "in window": any(q["evidence"].lower() in t.lower() for t in windowed),
                 "chars single": sum(map(len, single)), "chars window": sum(map(len, windowed))})
df = pd.DataFrame(rows); display(df); print(df[["in single chunk", "in window"]].mean())

,question,in single chunk,in window,chars single,chars window
0,How many dimensions does the all-MiniLM-L6-v2 ...,True,True,497,1489
1,What constant is used in the Reciprocal Rank F...,False,True,495,1487
2,Which reranker model is described as small and...,False,True,494,1486
3,What does HyDE stand for and how does it work?,True,True,498,1489
4,What is mean reciprocal rank?,False,False,494,1486
5,Which algorithm does Microsoft GraphRAG use to...,True,True,495,1483
6,What similarity threshold is suggested for a s...,True,True,493,1487
7,What is indirect prompt injection?,True,True,496,1118
8,What chunk size and overlap are a common start...,False,True,499,1489
9,How many candidates are typically passed to a ...,True,True,495,1487


in single chunk    0.80
in window          0.95
dtype: float64


## Window size vs context length
The window grows linearly in tokens; measure recall against cost.

In [5]:
rows = []
for w in [0, 1, 2, 3]:
    hit = []; chars = []
    for q in load_eval_questions():
        ctx = retrieve_with_window(q["question"], k=2, window=w)
        hit.append(any(q["evidence"].lower() in t.lower() for t in ctx)); chars.append(sum(map(len, ctx)))
    rows.append({"window": w, "recall": sum(hit) / len(hit), "avg_chars": sum(chars) / len(chars)})
pd.DataFrame(rows).set_index("window")

,recall,avg_chars
window,,
0,0.95,991.85
1,0.95,2448.35
2,1.00,3675.60
3,1.00,4696.25


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Fixes the 'answer is in the next chunk' failure | More tokens per query |
| No re-indexing; just a different read path | Overlapping chunks repeat text unless merged |
| Adjustable window | Can add distracting neighbours |

## When to use it

Narrative or explanatory text where ideas span paragraphs (articles, manuals, transcripts).

## Experiments to try

- Make the window adaptive: expand until a relevance threshold drops.
- De-duplicate overlapping text when merging neighbours.
- Compare window=1 with simply doubling the chunk size.

## Key takeaways

- Small chunks to match, bigger context to answer.
- Choose the window by measuring recall vs tokens.
- Keep document order.

---
**Next:** [13. Semantic Chunking](../13_semantic_chunking/semantic_chunking.ipynb)